# 🎬 MiniMax Face Refine (SAM Jawline Contour) - Interactive Web UI
### Method 2: SAM Semantic Jawline Segmentation
* **Description:** Segment-Anything semantic contour segmentation to isolate jawlines, collars, sunglasses, and props without color bleeding.
* **Accelerator:** Kaggle Dual Nvidia Tesla T4 GPU (or P100)
* **Mode:** **Interactive ComfyUI Web UI** (Cloudflare Tunnel)


In [ ]:
# [Step 1/4] Install ComfyUI & Custom Nodes
import os, sys, subprocess, time

try:
    sys.stdout.reconfigure(line_buffering=True)
    sys.stderr.reconfigure(line_buffering=True)
except Exception:
    pass

print(">>> [1/4] Installing ComfyUI & Face Refine Custom Nodes...", flush=True)

subprocess.run("wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared", shell=True, check=True)

if not os.path.exists("ComfyUI"):
    subprocess.run("git clone --depth 1 https://github.com/comfyanonymous/ComfyUI.git", shell=True, check=True)

custom_nodes = "ComfyUI/custom_nodes"
nodes = [
    "https://github.com/ltdrdata/ComfyUI-Manager.git",
    "https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git",
    "https://github.com/storyicon/comfyui_segment_anything.git",
    "https://github.com/fearnworks/comfyui_fearnworksnodes.git"
]
for n in nodes:
    target = os.path.join(custom_nodes, os.path.splitext(os.path.basename(n))[0])
    if not os.path.exists(target):
        subprocess.run(f"git clone --depth 1 {n} {target}", shell=True, check=True)

pkgs = ["huggingface_hub>=0.28.0", "hf_transfer", "opencv-python-headless", "ultralytics", "imageio", "imageio-ffmpeg"]
subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + pkgs, check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "ComfyUI/requirements.txt"], check=True)
print("✓ Environment ready.", flush=True)


In [ ]:
# [Step 2/4] Provision Model Weights (0-Sec Mount + HF Fallback)
import os, shutil, glob
from huggingface_hub import hf_hub_download

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.makedirs("ComfyUI/models/diffusion_models", exist_ok=True)
os.makedirs("ComfyUI/models/text_encoders", exist_ok=True)
os.makedirs("ComfyUI/models/vae", exist_ok=True)
os.makedirs("ComfyUI/models/sams", exist_ok=True)
os.makedirs("ComfyUI/models/ultralytics/bbox", exist_ok=True)

ds_candidates = glob.glob('/kaggle/input/*minimax*weights*')
ds_root = ds_candidates[0] if ds_candidates else '/kaggle/input/minimax-h3-open-weights'

def link_or_fetch(rel_ds, dst, hf_repo, hf_file):
    src = os.path.join(ds_root, rel_ds)
    if os.path.exists(src):
        if not os.path.exists(dst):
            try: os.symlink(src, dst)
            except Exception: shutil.copyfile(src, dst)
        print(f"✓ [MOUNTED] {dst}", flush=True)
    else:
        if not os.path.exists(dst):
            print(f"⏳ Downloading: {hf_file}...", flush=True)
            p = hf_hub_download(repo_id=hf_repo, filename=hf_file)
            try: os.symlink(p, dst)
            except Exception: shutil.copyfile(p, dst)

link_or_fetch("clip/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors", "ComfyUI/models/text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors", "Comfy-Org/MiniMax-H3", "text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors")
link_or_fetch("vae/minimax_h3_video_vae_int8_convrot.safetensors", "ComfyUI/models/vae/minimax_h3_video_vae_fp16.safetensors", "Comfy-Org/MiniMax-H3", "vae/minimax_h3_video_vae_fp16.safetensors")
link_or_fetch("diffusion_models/minimax_h3_fl2va_pruned_int8_convrot.safetensors", "ComfyUI/models/diffusion_models/minimax_h3_fl2va_pruned_int8_convrot.safetensors", "Comfy-Org/MiniMax-H3", "diffusion_models/minimax_h3_fl2va_pruned_int8_convrot.safetensors")

# SAM and YOLO models
sam_dst = "ComfyUI/models/sams/sam_vit_b.0_1.pth"
link_or_fetch("sams/sam_vit_b.0_1.pth", sam_dst, "fearnworks/vosr2", "sam_vit_b.0_1.pth")
if not os.path.exists(sam_dst):
    subprocess.run(f"wget -q -c https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth -O {sam_dst}", shell=True)

yolo_dst = "ComfyUI/models/ultralytics/bbox/face_yolov8m.pt"
if not os.path.exists(yolo_dst):
    subprocess.run(f"wget -q -c https://github.com/akanametov/yolo-face/releases/download/v0.0.0/face_yolov8m.pt -O {yolo_dst}", shell=True)
print("✓ Checkpoints mounted and verified.", flush=True)


In [ ]:
# [Step 3/4] Pre-load Workflows & Demo References
import os, shutil

wf_dir = "ComfyUI/user/default/workflows"
input_dir = "ComfyUI/input"
os.makedirs(wf_dir, exist_ok=True)
os.makedirs(input_dir, exist_ok=True)

local_wf = "workflows"
if os.path.exists(local_wf):
    for f in os.listdir(local_wf):
        p = os.path.join(local_wf, f)
        if f.endswith(".json"):
            shutil.copy(p, wf_dir)
        elif f.endswith(".mp4") or f.endswith(".png") or f.endswith(".jpg"):
            shutil.copy(p, input_dir)

print(f"✓ Deployed workflow: WF2_SAM_Contour_UI.json into ComfyUI Workflows menu!", flush=True)
print("✓ Sample assets deployed to ComfyUI/input/", flush=True)


In [ ]:
# [Step 4/4] Launch Interactive ComfyUI & Tunnel
import subprocess, threading, time, re

PORT = 8188
print("=" * 75, flush=True)
print(">>> [4/4] Starting ComfyUI Server & Cloudflare Tunnel...", flush=True)
print("=" * 75, flush=True)

comfy_proc = subprocess.Popen(
    f"{sys.executable} ComfyUI/main.py --listen 127.0.0.1 --port {PORT} --enable-cors-header",
    shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
)

def monitor():
    for line in comfy_proc.stdout:
        if any(w in line for w in ["Loaded", "Prompt", "Executing", "error", "Error"]):
            print(f"[ComfyUI] {line.strip()}", flush=True)

threading.Thread(target=monitor, daemon=True).start()

tunnel_proc = subprocess.Popen(f"/usr/local/bin/cloudflared tunnel --url http://127.0.0.1:{PORT}", shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
public_url = None
t_start = time.time()
while time.time() - t_start < 40:
    line = tunnel_proc.stdout.readline()
    if not line:
        time.sleep(0.5); continue
    m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if m:
        public_url = m.group(0); break

print("\n" + "=" * 80, flush=True)
if public_url:
    print(f"🌐 COMFYUI LIVE WEB UI: {public_url}", flush=True)
    print("👉 Open the link above, click Workflows menu, and select your pipeline!", flush=True)
print("=" * 80 + "\n", flush=True)

counter = 0
while True:
    time.sleep(60)
    counter += 1
    if counter % 5 == 0:
        print(f"[Heartbeat] Face Refine server is online ({counter} min).", flush=True)
